In [6]:
# LangGraph에서 상태 그래프를 만들기 위한 클래스와
# 그래프의 시작/끝을 나타내는 상수를 가져옵니다.
from langgraph.graph import StateGraph, START, END

# 딕셔너리 형태의 타입(키/값 타입을 명시한 dict)을
# 정의할 수 있게 해주는 도구입니다.
from typing_extensions import TypedDict

# 노드의 "캐시 정책"을 설정할 때 사용합니다.
# 예: 결과를 얼마나 오래 캐시에 보관할지(ttl) 등을 지정할 수 있습니다.
from langgraph.types import CachePolicy

# 메모리 안에 캐시를 저장하는 구현체입니다.
# 그래프를 컴파일할 때 이 캐시를 연결해 주면,
# 캐시 정책이 설정된 노드의 결과를 저장·재사용할 수 있습니다.
from langgraph.cache.memory import InMemoryCache

# 현재 시각을 가져오기 위해 사용합니다.
from datetime import datetime


In [7]:
# 그래프에서 오고 가는 상태의 모양을 정의합니다.
# time: 현재 시각을 담는 문자열 필드입니다.
class State(TypedDict):
  time: str

# 위에서 정의한 State 타입을 사용하는 상태 그래프를 만듭니다.
graph_builder = StateGraph(State)


In [8]:
# node_one: 아무 것도 하지 않는 노드입니다.
# 빈 dict 를 반환하면 상태가 그대로 유지됩니다.
def node_one(state: State):
  # print("node_one", state)
  return {}

# node_two: 현재 시각을 state 의 time 필드에 넣어 반환합니다.
# 이 노드에 캐시 정책을 적용하면, 같은 입력일 때
# 매번 새로 실행하지 않고 이전 결과를 재사용할 수 있습니다.
def node_two(state: State):
  # print("node_two", state)
  return {
    "time": f"{datetime.now()}"
  } 

# node_three: 아무 것도 하지 않는 노드입니다.
def node_three(state: State):
  # print("node_three", state)
  return {}
  

In [9]:
# 그래프에 노드를 등록합니다.
graph_builder.add_node("node_one", node_one)

# node_two 에만 캐시 정책을 적용합니다.
# cache_policy=CachePolicy(ttl=20) 의 의미:
#   - ttl(Time To Live) = 20초
#   - 이 노드의 결과를 20초 동안 캐시에 보관하고,
#     같은 입력으로 다시 호출되면 20초가 지나기 전까지는
#     노드를 다시 실행하지 않고 캐시된 결과를 그대로 씁니다.
graph_builder.add_node(
  "node_two",
  node_two,
  cache_policy=CachePolicy(ttl=20)
)
graph_builder.add_node("node_three", node_three)

# 노드들 사이의 실행 순서를 간선(edge)으로 정의합니다.
# START -> node_one -> node_two -> node_three -> END
graph_builder.add_edge(START, "node_one")
graph_builder.add_edge("node_one", "node_two")
graph_builder.add_edge("node_two", "node_three")
graph_builder.add_edge("node_three", END)

In [10]:
import time

# InMemoryCache 를 연결해서 그래프를 컴파일합니다.
# 이렇게 하면 cache_policy 가 설정된 노드(node_two)의 결과가
# 메모리 캐시에 저장되고, ttl 이 지나기 전까지 재사용됩니다.
graph = graph_builder.compile(cache=InMemoryCache())

# 5초 간격으로 graph.invoke({}) 를 6번 호출합니다.
# - 처음 4번(0~20초): node_two 의 ttl 이 20초이므로,
#   같은 입력({})에 대해 캐시된 결과를 그대로 씁니다.
#   => time 값이 매번 같게 나옵니다.
# - 5번째 호출(25초): 20초가 지나서 캐시가 만료되었으므로
#   node_two 가 다시 실행되고, 새로운 시각이 반환됩니다.
# - 6번째 호출(30초): 또 20초가 지나지 않았으므로
#   방금 캐시된 결과를 그대로 씁니다.
print(graph.invoke({}))
time.sleep(5)

print(graph.invoke({}))
time.sleep(5)

print(graph.invoke({}))
time.sleep(5)

print(graph.invoke({}))
time.sleep(5)

print(graph.invoke({}))
time.sleep(5)

print(graph.invoke({}))
time.sleep(5)

{'time': '2026-02-28 22:16:48.865507'}
{'time': '2026-02-28 22:16:48.865507'}
{'time': '2026-02-28 22:16:48.865507'}
{'time': '2026-02-28 22:16:48.865507'}
{'time': '2026-02-28 22:17:08.878342'}
{'time': '2026-02-28 22:17:08.878342'}
